# 03 — El producto end-to-end: del scraping a la dirección del dólar

Esta notebook recorre **todo el sistema, en orden y sin saltarse nada**:

| § | Etapa | Qué verás |
|---|---|---|
| 1 | **Scraping de noticias** (Bronze→Silver) | Los titulares crudos del día, sin LLM |
| 2 | **El movimiento de las monedas** | TRM oficial + Brent, DXY y bolsa colombiana |
| 3 | **GOLD: los topics y la golden query** | Cada noticia clasificada (topic, canal FX, severidad, entidades) y la consulta que filtra el oro del ruido |
| 4 | **El grafo de noticias** | Tópicos ↔ entidades con comunidades — los clusters del día |
| 5 | **El sistema multiagéntico** | QUÉ patrón es (la respuesta explícita) + el grafo LangGraph **renderizado** |
| 6 | **La corrida y la predicción** | El grafo completo ejecutándose → `DirectionalCall` |
| 7 | **Medición** | La tabla `predictions` que califica al sistema contra la realidad |

**El objetivo de estudio** (invariante desde el día 1): clasificar la **dirección** del
USD/COP — `down`/`up`/`neutral` — cruzando noticias analizadas por agentes con la señal
de la serie de tiempo, bajo una capa de racionalidad que obliga al contra-argumento y
acota la confianza por contrato.

---

### Actualizacion de producto: coherencia del adjudicador

El pipeline ya no confia solo en la prosa del LLM adjudicador. El juez debe declarar `dominant_signal` (`news`, `timeseries`, `market`, `none`) y el codigo valida que la direccion final coincida con esa senal. Si el racional dice que domina una noticia fiscal pero la direccion emitida apunta al lado contrario, el sistema corrige la direccion, baja/confina la confianza y deja el motivo en `consistency_notes`.

Los prompts tambien fueron endurecidos: cada agente debe trabajar como analista independiente, rechazar ruido, distinguir sorpresa de repeticion, nombrar canal de transmision FX y explicar por que una senal pierde contra otra.

In [ ]:
# Setup — limpio: sin os.chdir, sin adivinar rutas con cwd.
# cop_fx.paths resuelve la raíz desde el paquete y Settings lee el .env absoluto.
import sys

try:
    import cop_fx  # noqa: F401
except ModuleNotFoundError:
    raise RuntimeError(
        f"KERNEL EQUIVOCADO ({sys.executable}).\n"
        "En VS Code: selector de kernel → 'Python (cop-fx-intelligence)'."
    ) from None

from cop_fx.config.settings import get_settings
from cop_fx.paths import DATA_DIR, PROJECT_ROOT

settings = get_settings()
assert settings.openai_api_key is not None, f"Falta OPENAI_API_KEY en {PROJECT_ROOT / '.env'}"

print(f"✓ Proyecto : {PROJECT_ROOT.name}")
print(f"✓ LLM      : {settings.llm_model} ({settings.llm_provider})")
print(f"✓ Datos    : {DATA_DIR}")

## 1. Scraping de noticias — Bronze → Silver

Determinista y aburrido a propósito: feeds RSS de economía colombiana, CNN Colombia
(HTML de respaldo) y los feeds macro del lado USD (CNBC, MarketWatch). Cero LLM.
El resultado es `list[Article]` — el Silver que alimenta a los agentes.

In [ ]:
import pandas as pd

from cop_fx.data.news_fetcher import NewsFetcher

articles = NewsFetcher().fetch()
df_silver = pd.DataFrame(
    {
        "fecha": [a.published_at.strftime("%Y-%m-%d") for a in articles],
        "fuente": [a.source[:24] for a in articles],
        "titular": [a.title for a in articles],
    }
)
print(f"✓ {len(articles)} artículos SILVER — ⚠ esta tabla muestra SOLO titulares")
print("  (así llega el crudo; el TEXTO COMPLETO se descarga y analiza en la sección 3)")
print("  Orden: por FECHA con fuentes intercaladas — mismo día = mismo pie.\n")
df_silver.head(15)

## 2. El movimiento de las monedas

La TRM oficial (datos.gov.co) más el contexto de mercado que el estudio de la
notebook 02 validó: Brent, DXY y la bolsa colombiana (GXG) — recuerda que el equity
de ayer es el único predictor adelantado que pasó el filtro lead-lag.

In [ ]:
import matplotlib.pyplot as plt

from cop_fx.data.fx_fetcher import FXFetcher
from cop_fx.data.market_fetcher import fetch_market_panel

fx = FXFetcher().fetch()
panel = fetch_market_panel(fx)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(fx["ds"], fx["y"], color="#2c7fb8", lw=1.2)
axes[0].set_title(f"USD/COP (TRM oficial) — última: {fx['y'].iloc[-1]:,.2f}")
axes[0].grid(alpha=0.3)
for col in [c for c in panel.columns if c != "ds"]:
    axes[1].plot(panel["ds"], 100 * panel[col] / panel[col].iloc[0], label=col, lw=1.1)
axes[1].legend(); axes[1].grid(alpha=0.3)
axes[1].set_title("TRM vs Brent, DXY y bolsa CO — base 100")
plt.tight_layout(); plt.show()

## 3. GOLD — los topics y la *golden query*

Aquí el LLM toca el dato por primera vez. El `NewsAnalyzer` (el mismo del pipeline)
clasifica cada artículo con la **taxonomía de dos niveles**:

- `topic`: qué ES la noticia (15 dominios: `political_risk`, `monetary_policy`,
  `public_health`, `environment_climate`, `sports`...)
- `fx_relevance` + `fx_channel`: si transmite al dólar y POR QUÉ mecanismo —
  con coherencia forzada por contrato (deportes ⇒ relevancia `none` ⇒ pesa 0).

In [ ]:
from cop_fx.analysis.news_analyzer import NewsAnalyzer
from cop_fx.config.settings import get_settings
from cop_fx.data.article_body import attach_bodies

# Selección GOLD — en el ORDEN del fetcher (fecha + fuentes intercaladas):
# se descarga el texto completo y se FILTRA lo que no tiene texto analizable.
# NO se reordena por longitud: extraíble ≠ relevante (la reforma tributaria
# de Portafolio vale aunque su paywall solo deje una línea de resumen).
cfg = get_settings()
pool = articles[:100]
attach_bodies(pool, max_articles=100)

def _text_len(a):
    return len(getattr(a, "body", "") or a.summary)

subset = [a for a in pool if _text_len(a) >= cfg.min_analyzable_chars][:20]
print(f"Pool {len(pool)} → descartados {len(pool) - sum(1 for a in pool if _text_len(a) >= cfg.min_analyzable_chars)} sin texto → analizamos {len(subset)}")

analysis = NewsAnalyzer().analyze(subset)

pd.set_option("display.max_colwidth", 90)
df_gold = pd.DataFrame([
    {
        "titular": it.article.title[:55],
        "autor": getattr(it.article, "author", "") or "(sin autor)",
        "fuente": it.article.source[:14],
        "noticia": ((getattr(it.article, "body", "") or it.article.summary)[:85] + "…"),
        "chars_texto": _text_len(it.article),
        "topic": it.topic,
        "fx_relevance": it.fx_relevance,
        "fx_channel": it.fx_channel,
        "severity": it.severity,
        "bullish_cop": it.bullish_cop,
    }
    for it in analysis.items
])
print(f"✓ {len(df_gold)} GOLD · fuentes: {sorted(df_gold['fuente'].unique())} · "
      f"texto medio {df_gold['chars_texto'].mean():,.0f} chars\n")
print(f"Narrativa del día:\n{analysis.narrative}\n")
df_gold

In [ ]:
# EVIDENCIA — esto es literalmente lo que lee el agente (no el titular):
for a in subset[:3]:
    text = (getattr(a, "body", "") or a.summary)[:350]
    print(f"━━ [{a.source[:20]}] {a.title[:70]}")
    print(f"   TEXTO ({_text_len(a)} chars): «{text}…»\n")

In [ ]:
# LA GOLDEN QUERY: separar el oro del ruido.
# De todo lo scrapeado, ¿qué tiene canal de transmisión al dólar y con qué fuerza?
golden = (
    df_gold[df_gold["fx_relevance"] != "none"]
    .sort_values("severity", key=lambda s: s.map({"high": 0, "medium": 1, "low": 2}))
    .reset_index(drop=True)
)
print(f"Golden query: {len(golden)} de {len(df_gold)} artículos tienen señal FX\n")
display(golden)

print("\nTopic × relevancia FX — cuánto del día es señal vs ruido:")
pd.crosstab(df_gold["topic"], df_gold["fx_relevance"])

## Keywords, entidades y taxonomía normalizada

La capa GOLD no solo clasifica `topic`, `fx_channel` y severidad. También extrae `keywords` y `entities`, que sirven para auditar qué conceptos están dominando la señal noticiosa. En producto, el dashboard pondera estas palabras por importancia (`severity x relevance + canal`) para evitar que una palabra repetida en noticias de baja relevancia parezca más importante que un shock material.

La normalización de topics vive en `cop_fx.analysis.topic_taxonomy`: agrupa dominios finos en familias de decisión (`Riesgo pais`, `Riesgo fiscal`, `Commodities`, etc.) y marca huecos de investigación cuando una noticia material cae en `other`, no tiene canal FX, o parece un falso positivo.


In [ ]:
from cop_fx.analysis.topic_taxonomy import article_importance, normalize_topic

if "df" in globals() and {"topic", "keywords", "fx_relevance", "fx_channel", "severity"}.issubset(df.columns):
    df_keywords = df.copy()
    df_keywords["topic_family"] = df_keywords["topic"].apply(normalize_topic)
    df_keywords["importance"] = df_keywords.apply(article_importance, axis=1)
    display(df_keywords[["title", "topic_family", "keywords", "entities", "importance"]].head(20))
else:
    print("Ejecuta primero la celda que construye df GOLD para ver keywords ponderadas.")


## 4. El grafo de noticias — tópicos ↔ entidades

Tu idea original del proyecto: ver cómo interactúan los temas del día. Grafo bipartito
(tópicos ↔ entidades nombradas), aristas ponderadas por co-mención, **comunidades**
detectadas por modularidad. Cada comunidad es un cluster temático — exactamente la
unidad de trabajo que el orchestrator despacha con `Send` en la sección siguiente.

In [ ]:
import networkx as nx
from networkx.algorithms.community import greedy_modularity_communities

G = nx.Graph()
for it in analysis.items:
    t = f"◼ {it.topic}"
    G.add_node(t, kind="topic")
    for e in it.entities:
        en = f"● {e}"
        G.add_node(en, kind="entity")
        w = (G.get_edge_data(t, en) or {}).get("weight", 0)
        G.add_edge(t, en, weight=w + 1)

communities = list(greedy_modularity_communities(G, weight="weight"))
bridges = [n for n in G.nodes if G.nodes[n]["kind"] == "entity"
           and len({nb for nb in G.neighbors(n) if G.nodes[nb]["kind"] == "topic"}) > 1]
print(f"{G.number_of_nodes()} nodos · {G.number_of_edges()} aristas · "
      f"{len(communities)} comunidades · puentes entre tópicos: {bridges or 'ninguno'}")

fig, ax = plt.subplots(figsize=(13, 8))
pos = nx.spring_layout(G, k=0.65, seed=42, weight="weight")
palette = plt.colormaps["tab10"].colors
colors = [palette[next(ci for ci, c in enumerate(communities) if n in c) % 10] for n in G.nodes]
sizes = [900 + 250 * int(G.degree(n)) if G.nodes[n]["kind"] == "topic"
         else 120 + 90 * int(G.degree(n)) for n in G.nodes]
nx.draw_networkx_edges(G, pos, alpha=0.3, ax=ax)
nx.draw_networkx_nodes(G, pos, node_color=colors, node_size=sizes, alpha=0.85, ax=ax)
nx.draw_networkx_labels(G, pos, font_size=7, ax=ax)
ax.set_title("Grafo de noticias: tópicos (◼) ↔ entidades (●) — color = comunidad = cluster del Send")
ax.axis("off"); plt.tight_layout(); plt.show()

## 5. El sistema multiagéntico — la respuesta explícita: ¿qué patrón es?

**No es UN patrón: es una composición de seis, cada uno resolviendo un problema
distinto.** El dominante es **orchestrator-workers + evaluator**; los demás son capas
de soporte. Este era el dilema original del proyecto ("¿prompt chaining, routing,
parallelization u orchestrator-worker?") y esta es la respuesta final:

| Patrón | Nodo(s) en este sistema | Qué problema resuelve |
|---|---|---|
| **Parallelization** | `fetch_fx` ‖ `fetch_market` ‖ `fetch_news` desde START; los N workers en un mismo superstep | Las señales son independientes: descargarlas y analizarlas a la vez |
| **Routing** | `check_materiality` → `add_conditional_edges` → `orchestrate` o `skip_news` | Control de costo: un día sin noticia material no gasta tokens de análisis |
| **Orchestrator-workers** (API `Send`) | `orchestrate` → `Send("topic_worker", cluster)` × N → reducers `operator.add` | La cantidad de analistas la decide el DATO (los clusters del §4), no el código — un grafo estático no puede |
| **Prompt chaining** | dentro de cada `topic_worker`: extraer → clasificar → canal de transmisión → impacto | Sub-pasos secuenciales de razonamiento por cluster |
| **Evaluator / racionalidad** | `adjudicate` (`defer=True`, tier judge) | Reconcilia noticias vs serie vs contexto de mercado; abogado del diablo obligatorio; confianza ACOTADA por validadores Pydantic, no por el prompt |
| **Persistencia + HITL + memoria** | `human_review` (`interrupt`) · `load_memory` · `SqliteSaver` por `thread_id` | Pausa para aprobación humana antes de publicar y reanuda en otro proceso; el adjudicador recuerda su track-record entre corridas |

**Las dos reglas de oro del diseño:**
1. El LLM solo razona sobre TEXTO; todos los NÚMEROS (señales, pesos, confianza,
   evaluación) los produce código determinista. El modelo no puede inventarse un score.
2. Un nodo nuevo solo entra si toma una decisión que un nodo determinista no podía
   tomar — el multiagente se gana con necesidad, no con ambición.

## Etapa 6 — las dos memorias de LangGraph + human-in-the-loop

El grafo usa los dos tipos de memoria que LangGraph distingue, más un `interrupt`:

- **Checkpointer (memoria de corto plazo, por `thread_id`)**: `SqliteSaver` guarda
  el estado del grafo paso a paso. Es lo que habilita **pausar y reanudar en otro
  proceso**. Sin checkpointer no hay dónde "congelar" la corrida.
- **Store (memoria de largo plazo, entre hilos)**: un almacén
  `namespace → key → value`. El nodo `load_memory` publica ahí el track-record; la
  fuente durable real es `predictions.db`.
- **`interrupt` (HITL)**: el nodo `human_review` lanza `interrupt(payload)` antes de
  publicar; el grafo se pausa y devuelve el tweet propuesto. Se reanuda con
  `Command(resume={"approved": bool})` sobre el mismo `thread_id` — **sin recomputar
  el LLM**, porque el checkpointer restaura el estado.

La **memoria entre corridas** cierra el loop de aprendizaje: el adjudicador lee qué
llamó antes y si acertó, y se calibra ("tus últimas llamadas de alta confianza
fallaron — exige más evidencia") en vez de empezar de cero cada día. `load_memory`
corre como rama paralela SIN arista al adjudicador, igual que `fetch_market`: escribe
estado que el nodo `defer=True` espera, sin sumar un trigger que lo dispararía dos veces.

```bash
uv run cop-fx run --review --publish            # corre y PAUSA pidiendo aprobación
uv run cop-fx resume --thread <fecha> --approve # reanuda y publica
```

In [ ]:
# El grafo LangGraph, RENDERIZADO (no texto): éste es el sistema completo
from IPython.display import Image, display

from cop_fx.agents.graph import build_graph

compiled = build_graph().compile()
try:
    display(Image(compiled.get_graph().draw_mermaid_png()))
except Exception as exc:
    print(f"(render remoto no disponible: {exc} — versión texto)\n")
    print(compiled.get_graph().draw_mermaid())

## 6. La corrida completa → la predicción de la dirección

Todo lo anterior, orquestado en una sola invocación — lo mismo que ejecuta
`uv run cop-fx run` cada día.

In [ ]:
from cop_fx.agents.graph import run_pipeline

final_state = run_pipeline()

print("\n=== resumen de la corrida ===")
print(f"TRM            : {final_state.get('latest_rate', 0):,.2f} ({final_state.get('rate_change_pct', 0):+.2f}% 30d)")
print(f"Mercado (ayer) : {final_state.get('market_signal', {})}")
print(f"¿Día material? : {final_state.get('has_material_news')}")
print(f"Clusters (Send): { {t: len(i) for t, i in final_state.get('clusters', {}).items()} }")
print(f"Errores        : {final_state.get('errors', [])}")

In [ ]:
call = final_state["directional_call"]
ARROW = {"down": "⬇️ USD/COP BAJA (COP se fortalece)",
         "up": "⬆️ USD/COP SUBE (COP se debilita)",
         "neutral": "⏸️ NEUTRAL — abstención"}

print("=" * 70)
print(f"  {ARROW[call['direction']]}")
print(f"  confianza {call['confidence']:.2f} · horizonte {call['horizon_days']}d · reconciliación: {call['reconciliation']}")
print("=" * 70)
print(f"  noticias : {call['news_signal']['direction']} (score {call['news_signal']['score']})")
print(f"  serie    : {call['ts_signal']['direction']} ({call['ts_signal']['yhat_delta_pct']:+.2f}%)")
print(f"  mercado  : {final_state.get('market_signal', {}).get('direction', 'n/d')} (contexto, no voto)")
print("-" * 70)
print(f"RACIONAL:\n{call['rationale']}\n")
print(f"ABOGADO DEL DIABLO:\n{call['devils_advocate']}")

top = final_state.get("top_story") or {}
if top:
    print("\n" + "=" * 70)
    print(f"📌 NOTICIA MÁS IMPORTANTE DEL DÍA (agente editor)")
    print(f"   {top['title']}  ({top['source']})")
    print(f"   Por qué importa: {top['why_it_matters'][:300]}")
    print(f"   Vigilar: {top['watch_next'][:200]}")

### El reporte diario completo

Lo que el sistema persiste en `reports/` — incluye la **📌 Noticia del día** con su justificación, el contexto de mercado, el racional y el abogado del diablo.

In [ ]:
from IPython.display import Markdown, display

display(Markdown(final_state["report_markdown"]))

## 7. Medición — el sistema rinde cuentas

Cada corrida deja su predicción en `predictions`; al vencer el horizonte se califica
sola contra la TRM real. Hit-rate, matriz de confusión y accuracy-por-confianza son
las métricas (dirección, no MAPE). El backtest de la serie vive en la notebook 02.

In [ ]:
from cop_fx.tracking import PredictionStore

preds = PredictionStore().all()
print(f"{len(preds)} predicciones registradas")
preds[["run_date", "direction", "confidence", "reconciliation",
       "news_direction", "ts_direction", "latest_rate", "actual_direction", "hit"]]

In [ ]:
# Conclusión GENERADA de esta corrida
ns, ts = call["news_signal"], call["ts_signal"]
mkt = final_state.get("market_signal", {})
print(f"""
┌─ CONCLUSIÓN DE LA CORRIDA ─────────────────────────────────────────
│ {len(final_state.get('raw_articles', []))} titulares → día {'MATERIAL' if final_state.get('has_material_news') else 'no material'} → {len(final_state.get('clusters', {}))} analistas (Send) → {len(final_state.get('analyzed_articles', []))} artículos GOLD
│
│ noticias: {ns['direction']:>7} (score {ns['score']:+.2f})   serie: {ts['direction']:>7} ({ts['yhat_delta_pct']:+.2f}%)
│ mercado : {mkt.get('direction', 'n/d'):>7} (equity ayer {mkt.get('equity_ret_1d_pct', 0):+.2f}%)
│
│ VEREDICTO: USD/COP {call['direction'].upper()} a {call['horizon_days']} días · confianza {call['confidence']:.2f} · {call['reconciliation']}
│ Registrado en data/predictions.db — se autocalifica al vencer el horizonte.
└────────────────────────────────────────────────────────────────────
""")

---
## Cierre

Acabas de ver el sistema completo: **scraping → monedas → golden query (topics) →
grafo de noticias → sistema multiagéntico (routing + Send + chaining + evaluator) →
predicción de la dirección → medición → memoria entre corridas + aprobación humana (HITL)**. Los otros dos cuadernos profundizan las
patas: `01_noticias` (la ingesta y la taxonomía) y `02_series_de_tiempo` (el
diagnóstico cuantitativo y qué activos se ganaron su lugar).

**Para dominar LangGraph replicando esto**: reconstruye el grafo en una notebook
vacía en este orden — (1) un nodo con `with_structured_output`, (2) el router con
`add_conditional_edges`, (3) el fan-out con `Send` + reducers, (4) el join con
`defer=True` y un solo trigger, (5) la persistencia con un checkpointer (`SqliteSaver`) +
`interrupt` para human-in-the-loop, reanudando con `Command(resume=...)`. Verifícate contra `tests/unit/test_graph_nodes.py`.

## 7. Auditoria del ultimo run

Esta celda carga la ultima prediccion persistida y el reporte mas reciente. Sirve para revisar tres cosas despues de correr el flujo completo: direccion final, senal dominante y checks de coherencia. Si `consistency_notes` contiene correcciones, el sistema detecto que el LLM habia emitido algo inconsistente y lo dejo trazado.

In [ ]:
from pathlib import Path
import sqlite3
import pandas as pd

from cop_fx.paths import DATA_DIR, REPORTS_DIR

pred_db = DATA_DIR / "predictions.db"
if pred_db.exists():
    with sqlite3.connect(pred_db) as conn:
        latest = pd.read_sql(
            "SELECT * FROM predictions ORDER BY run_date DESC LIMIT 1", conn
        )
    display(latest.T)
else:
    print("Sin predictions.db todavia. Corre: uv run cop-fx run")

reports = sorted(REPORTS_DIR.glob("report_*.md"))
if reports:
    report = reports[-1]
    print(f"Reporte: {report}")
    print("
".join(report.read_text(encoding="utf-8").splitlines()[:45]))
else:
    print("Sin reportes todavia.")

## Resultado auditado de la corrida completa (2026-06-12)

Con los prompts endurecidos y el validador de coherencia activo, la corrida completa produjo:

- **Directional Call:** `neutral`, confianza `0.44`.
- **Reconciliacion:** `partial`.
- **dominant_signal:** `none`.
- **Noticias:** `neutral` (`score = 0.05`).
- **Serie:** `down` (`-0.45%`), pero Prophet y ARIMA no coinciden.
- **Mercado:** `down` por equity colombiano fuerte, usado solo como contexto/tie-breaker.
- **Noticia clave:** ajuste al alza de meta de deficit fiscal e inflacion por Hacienda.

Lectura de producto: esta es una mejora respecto al comportamiento anterior. El sistema ya no fuerza una direccion cuando el racional reconoce conflicto. La abstencion es coherente porque el titular fiscal es relevante para riesgo pais, pero el agregado de noticias queda casi plano y la senal cuantitativa no tiene consenso interno.

Siguiente cuello de botella observado: calidad de texto fuente. Varias URLs de Portafolio devuelven cuerpos demasiado cortos por paywall/boilerplate, asi que algunos analisis dependen del summary RSS. Eso limita severidad, top-story selection y trazabilidad. La siguiente mejora de resultados deberia ser una capa de calidad de fuente: `body_quality`, penalizacion por paywall, y/o reemplazo de articulos truncados por otra fuente que cubra el mismo evento.

## Co-ocurrencia pairwise de keywords

Inspirado en el flujo `tidytext` (`unnest_tokens` → `pairwise_cor`), este proyecto usa una versión Python sobre la capa GOLD: cada noticia es un documento, cada keyword juzgada es una variable booleana, y cada par recibe coeficiente phi. Antes de correlacionar, se eliminan stopwords de dominio y el LLM decide si el término es accionable para USD/COP. Esto evita que palabras como `Colombia` dominen por frecuencia sin aportar señal.


In [ ]:
from cop_fx.analysis.keyword_analysis import (
    build_term_documents,
    judge_terms,
    pairwise_phi,
    rank_terms,
)
from cop_fx.config.settings import get_settings

settings = get_settings()

if "df_keywords" in globals():
    gold_terms = build_term_documents(
        df_keywords[df_keywords["fx_relevance"] != "none"],
        domain_stopwords=settings.keyword_domain_stopwords,
        include_entities=False,
    )
    candidates = (
        gold_terms.groupby(["term_key", "term"], as_index=False)
        .agg(score=("importance", "sum"), noticias=("doc_id", "nunique"))
        .sort_values(["score", "noticias"], ascending=False)
        .head(settings.keyword_top_n * 2)["term"]
        .tolist()
    )
    decisions = judge_terms(
        candidates,
        use_llm=settings.keyword_llm_judge_enabled,
        domain_stopwords=settings.keyword_domain_stopwords,
    )
    ranked_terms = rank_terms(
        gold_terms,
        decisions,
        min_articles=settings.keyword_min_articles,
        top_n=settings.keyword_top_n,
    )
    pairs = pairwise_phi(
        gold_terms,
        ranked_terms["termino"].tolist(),
        min_joint=settings.keyword_pairwise_min_joint,
    )
    display(ranked_terms[["termino", "score", "noticias", "llm_score", "reason"]].head(15))
    display(pairs.head(15))
else:
    print("Ejecuta primero la sección de keywords para construir df_keywords.")
